In [11]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import MultinomialNB
import typing
import re


In [12]:
docs_train = [
  "LOL. I love this site http://scam.com/enter/to/win.",
  "My mother loves the lotto",
  "Dear Sir or Madam, you have been named in a will to receive a sum of $2000000",
  "Will you please stop using that site",
  "hot and lonely tortoises in your neighborhood XXX",
  "It's lonely being an outlier",
  "Announcing the WaffleCOIN ICO",
  "Important message from the IRS.  Send your bank account number and SSN immediately to irs-staff@scam.com"
]

docs_test = [
  "OMG. Have you seen this site http://scam.com/p0wnd",
  "Dear Sir or Madam, I am contacting you about an urgent financial matter. I have taken possession of an abandoned account with a large sum of money ($1,000,000)",
  "Did you remember to buy lettuce for the tortoise?",
  "UPCOMING ICO! Get ready for MuffinCOIN",
]


In [13]:
y_train = [
  "SPAM",
  "HAM",
  "SPAM",
  "HAM",
  "SPAM",
  "HAM",
  "SPAM",
  "SPAM"
]

y_test = [
  "SPAM",
  "SPAM",
  "HAM",
  "SPAM"
]

In [14]:
vectorizer = CountVectorizer(
  # case fold all text 
  # before generating n-grams
  lowercase=True,
  # optionally apply the specified function
  # before counting n-grams
  preprocessor=None,
  # optionally provide a list of tokens to remove/ignore before generating n-grams
  stop_words=None,
  # specify a range of n-grams as (min_n, max_n). 
  # (1, 1) means unigrams.
  # (1, 2) means unigrams and bigrams
  # (4, 5) means 4-grams and 5-grams
  ngram_range=(1, 1),
  # "word", "char" (character), or "char_wb" n-grams
  analyzer="word",
  # whether or not to use binary counts
  binary=False
)

In [15]:
vectorizer.fit(docs_train)

CountVectorizer()

In [44]:
vectorizer.vocabulary_

{'lol': 21,
 'love': 24,
 'this': 48,
 'site': 41,
 'http': 14,
 'scam': 38,
 'com': 8,
 'enter': 10,
 'to': 49,
 'win': 54,
 'my': 29,
 'mother': 28,
 'loves': 25,
 'the': 47,
 'lotto': 23,
 'dear': 9,
 'sir': 40,
 'or': 34,
 'madam': 26,
 'you': 56,
 'have': 12,
 'been': 6,
 'named': 30,
 'in': 18,
 'will': 53,
 'receive': 37,
 'sum': 45,
 'of': 33,
 '2000000': 0,
 'please': 36,
 'stop': 44,
 'using': 51,
 'that': 46,
 'hot': 13,
 'and': 3,
 'lonely': 22,
 'tortoises': 50,
 'your': 57,
 'neighborhood': 31,
 'xxx': 55,
 'it': 20,
 'being': 7,
 'an': 2,
 'outlier': 35,
 'announcing': 4,
 'wafflecoin': 52,
 'ico': 15,
 'important': 17,
 'message': 27,
 'from': 11,
 'irs': 19,
 'send': 39,
 'bank': 5,
 'account': 1,
 'number': 32,
 'ssn': 42,
 'immediately': 16,
 'staff': 43}

In [17]:
vectorizer.transform(
    [
        "It's going to cost you $23,030.12 or more.", 
        "Pay here: http://super-sketchy-site.info"
    ]
).todense()

matrix([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

In [19]:
i2v = dict((i, v) for (v, i) in vectorizer.vocabulary_.items())

In [20]:
i2v[0]

'2000000'

In [21]:
i2v[5]

'bank'

In [22]:
vectorizer.inverse_transform(
    vectorizer.transform(["It's going to cost you $23,030.12 or more."]).todense()
)

/usr/local/lib/python3.8/site-packages/sklearn/utils/validation.py:593: FutureWarning: np.matrix usage is deprecated in 1.0 and will raise a TypeError in 1.2. Please convert to a numpy array with np.asarray. For more information see: https://numpy.org/doc/stable/reference/generated/numpy.matrix.html
  warnings.warn(


[array(['it', 'or', 'to', 'you'], dtype='<U12')]

In [23]:
len(vectorizer.vocabulary_)

58

In [24]:
lbl_encoder = LabelEncoder()

In [25]:
lbl_encoder.fit(y_train)

LabelEncoder()

In [26]:
lbl_encoder.classes_

array(['HAM', 'SPAM'], dtype='<U4')

In [28]:
lbl_encoder.transform(["HAM"])

array([0])

In [29]:
clf = MultinomialNB(fit_prior=True)

In [30]:
clf.fit(vectorizer.transform(docs_train), lbl_encoder.transform(y_train))

MultinomialNB()

In [31]:
clf.classes_

array([0, 1])

In [33]:
yhats = clf.predict(vectorizer.transform(docs_test))

In [34]:
predictions = lbl_encoder.inverse_transform(yhats)

In [36]:
for pred, gold in zip(predictions, y_test):
    print(f"PRED: {pred} \tGOLD: {gold}")

PRED: SPAM 	GOLD: SPAM
PRED: SPAM 	GOLD: SPAM
PRED: SPAM 	GOLD: HAM
PRED: SPAM 	GOLD: SPAM


In [37]:
vectorizer.transform(["ZAMBORTANI DIEMPO"]).todense()

matrix([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

In [38]:
vectorizer.transform(["Kltpzyxm"]).todense()

matrix([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

In [39]:
vectorizer.transform(["$20.00"]).todense()


matrix([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])